In [ ]:
import os
import numpy as np
import xarray as xr
import zarr

import dask
import matplotlib.pyplot as plt

from dask.diagnostics import ProgressBar

In [ ]:
latlon = np.load('/project/home/p200177/DE_371/datasets/datasets_SMHI/256x256/latlon.npy')

In [ ]:
e = 0.1
min_lat, max_lat, min_lon, max_lon = (
    latlon[0].min() - e,
    latlon[0].max() + e,
    latlon[1].min() - e,
    latlon[1].max() + e
)

In [ ]:
print(min_lat, max_lat, min_lon, max_lon)

In [ ]:
output_path = '/project/home/p200177/u101329/DE371_bis/MEPS_subdomain'

In [ ]:
MEPS_path = '/project/home/p200177/DE_371/datasets/MEPS/aifs-meps-2.5km-2020-2023-1h-v2.zarr'

In [ ]:
pds = xr.open_zarr(MEPS_path, zarr_format=2)

In [ ]:
#pds.attrs["variables_metadata"].items()

In [ ]:
print(pds)

In [ ]:
cell_mask = (
    (pds.latitudes >= min_lat) &
    (pds.latitudes <= max_lat) &
    (pds.longitudes >= min_lon) &
    (pds.longitudes <= max_lon)
).compute()

In [ ]:
cell_idx = np.nonzero(cell_mask.values)[0]
cell_idx

In [ ]:
ds_sub = pds.isel(cell=cell_idx)

In [ ]:
print("Selected cells:", cell_idx.size)
print(ds_sub)

In [ ]:
def latlon_to_harmonie_grid(
    lon, lat,
    NLON=960,
    NLAT=1080,
    LONC=17.5,
    LATC=63.3,
    LON0=15.0,
    LAT0=63.3,
    GSIZE=2500.0,
    R=6371229
):
    """
    Convert lat/lon (degrees, 1D arrays) to HARMONIE grid indices (i, j)
    using Lambert Conformal Conic (1SP).
    """

    # --- Degrees to radians
    lon = np.deg2rad(lon)
    lat = np.deg2rad(lat)
    lon0 = np.deg2rad(LON0)
    lat0 = np.deg2rad(LAT0)
    lonc = np.deg2rad(LONC)
    latc = np.deg2rad(LATC)

    # --- LCC constants (1 standard parallel: LAT0)
    n = np.sin(lat0)
    F = (np.cos(lat0) * np.tan(np.pi / 4 + lat0 / 2) ** n) / n

    def rho(phi):
        return R * F / np.tan(np.pi / 4 + phi / 2) ** n

    # --- Projection of input points
    rho_p = rho(lat)
    theta = n * (lon - lon0)

    x = rho_p * np.sin(theta)
    y = rho(lat0) - rho_p * np.cos(theta)

    # --- Projection of grid center
    rho_c = rho(latc)
    theta_c = n * (lonc - lon0)

    xc = rho_c * np.sin(theta_c)
    yc = rho(lat0) - rho_c * np.cos(theta_c)

    # --- Convert meters → grid indices
    i = (x - xc) / GSIZE + (NLON - 1) / 2
    j = (y - yc) / GSIZE + (NLAT - 1) / 2

    return i, j

In [ ]:
ij = latlon_to_harmonie_grid(ds_sub["longitudes"],
    ds_sub["latitudes"])

In [ ]:
i = ij[0].compute()
j = ij[1].compute()
print(i.min(), i.max())
print(j.min(), j.max())

In [ ]:
inc = np.array(range(0,ds_sub.dims['cell']))
plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=inc,
    s=1
)
plt.colorbar(label="count")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Inc")
plt.show()

In [ ]:

plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=ds_sub["longitudes"],
    s=1
)
plt.colorbar(label="deg")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Lon")
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    i,
    j,
    c=ds_sub["latitudes"],
    s=1
)
plt.colorbar(label="deg")
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Lat")
plt.show()

In [ ]:
ifrac = i - np.floor(i)


In [ ]:
print(ifrac.max(),ifrac.min())

plt.hist(ifrac,bins=100)

In [ ]:
jfrac = j - np.floor(j)

In [ ]:
print(jfrac.max(),jfrac.min())

plt.hist(ifrac,bins=100)

In [ ]:
ii = np.ceil(i.data).astype('int') 
jj = np.ceil(j.data).astype('int')

In [ ]:
print(np.min(ii), np.max(ii))

In [ ]:
print(np.min(jj), np.max(jj))

In [ ]:
largest = np.zeros((577-292+1,530-219+1)).astype('bool')
print(largest.shape)

In [ ]:
for i in range(0,ds_sub.dims['cell']):
    largest[jj[i]-292,ii[i]-219] = True

In [ ]:
plt.figure(figsize=(8, 6))
plt.imshow(largest)
plt.show()

In [ ]:
mii = np.where( np.logical_and(ii>=(255), ii<(255+256)), True, False)

In [ ]:
mjj = np.where( np.logical_and(jj>=305 , jj<305+256), True, False)


In [ ]:
mask = np.logical_and(mii,mjj)

In [ ]:
np.unique(ii[mask]).shape

In [ ]:
ii[mask]

In [ ]:
jj[mask]

In [ ]:
np.unique(jj[mask]).shape

In [ ]:
indixes = np.array(range(0,ds_sub.dims['cell']))

filter_sub  = indixes[mask]

In [ ]:
filter_sub

In [ ]:
final_filter = cell_idx[filter_sub]

In [ ]:
final_filter

In [ ]:
final_ds = pds.isel(cell=final_filter)

In [ ]:
print("Selected cells:", final_filter.size)
print(final_ds)

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    final_ds["longitudes"],
    final_ds["latitudes"],
    c=final_ds["longitudes"],
    s=1
)
plt.colorbar(label="Wind speed")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Longtudes")
plt.show()

In [ ]:
output_path = '/project/home/p200177/u101329/DE371_bis/MEPS_subdomain'

In [ ]:
vars_keep = ["10u", "10v", "2t", "10si","tp"]
var_names = final_ds.attrs["variables"]

vidx = [var_names.index(v) for v in vars_keep]

In [ ]:
ds_reduced = final_ds.isel(time=slice(0, 24*60+1),variable=vidx)

In [ ]:
ds_reduced = ds_reduced.drop_vars(
    ["minimum", "maximum", "mean", "sums", "squares", "stdev"],
    errors="ignore"
)

In [ ]:
ds_reduced.attrs["variables"] = vars_keep

# Optionally trim variables_metadata
if "variables_metadata" in ds_reduced.attrs:
    ds_reduced.attrs["variables_metadata"] = {
        k: v
        for k, v in ds_reduced.attrs["variables_metadata"].items()
        if k in vars_keep
    }

In [ ]:
dims = ("time", "ensemble", "cell")

data = ds_reduced["data"]

ds_reduced["minimum"] = data.min(dim=dims)
ds_reduced["maximum"] = data.max(dim=dims)
ds_reduced["sums"]    = data.sum(dim=dims)
ds_reduced["mean"]    = data.mean(dim=dims)
ds_reduced["squares"] = (data ** 2).sum(dim=dims)
ds_reduced["stdev"]   = data.std(dim=dims)
 
 

In [ ]:
with ProgressBar():
    ds_reduced.to_zarr(
        f"{output_path}/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced.zarr",
        mode="w",
        zarr_format=2,
    )

In [ ]:
ds_red = xr.open_zarr(f"{output_path}/aifs-meps-2.5km-2020-2023-1h-v2_SMHI_subdomain_reduced.zarr", zarr_format=2)

In [ ]:
print(ds_red)

In [ ]:
var_names = ds_red.attrs["variables"]
v_idx = var_names.index("10u")
print(v_idx)
field = ds_red["data"].isel(
    time=100,
    variable=v_idx,
    ensemble=0
)

In [ ]:
ds_red.attrs["variables"]

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    ds_red["longitudes"],
    ds_red["latitudes"],
    c=field,
    s=1
)
plt.colorbar(label="Wind speed")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Longtudes")
plt.show()

In [ ]:
d2 = np.array(field.data)

In [ ]:
d2.shape = (256,256)

In [ ]:
plt.imshow(d2)